# Dia 1 — Preparacao de Dados e Analise Exploratoria (EDA)

**Autor:** Henry  
**Dataset:** Olist Brazilian E-Commerce  
**Objetivo:** Construir o DataFrame unificado de pedidos entregues, criar a variavel alvo `foi_atraso`, e analisar correlacoes de Pearson para identificar features promissoras para o modelo de ML.

---

## Task 1 — Carregar CSVs, Filtrar Pedidos Delivered e Realizar Merges

Carregamos 5 tabelas do dataset Olist e unificamos via left joins sequenciais.  
Apenas pedidos com `order_status == 'delivered'` sao mantidos.

In [ ]:
# Setup: imports e constantes
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

DATA_RAW = '../data/raw/'
print('Setup OK')

In [ ]:
# 1. Carregar orders
orders = pd.read_csv(DATA_RAW + 'olist_orders_dataset.csv')
print('orders.shape:', orders.shape)
orders.info()

In [ ]:
# 2. Filtrar delivered e dropar NaN em order_delivered_customer_date
df = orders[orders['order_status'] == 'delivered'].copy()
df = df.dropna(subset=['order_delivered_customer_date'])
print(f'Pedidos delivered (sem NaN na data de entrega): {df.shape[0]} rows')
print(f'Status unicos restantes: {df["order_status"].unique()}')

In [ ]:
# 3. Merge com customers (customer_id)
customers = pd.read_csv(DATA_RAW + 'olist_customers_dataset.csv')
df = df.merge(customers, on='customer_id', how='left')
print(f'Apos merge customers: {df.shape}')
print(f'Colunas adicionadas: customer_state, customer_zip_code_prefix')

In [ ]:
# 4. Merge com order_items (order_id) — rows aumentam (1 order = N items)
items = pd.read_csv(DATA_RAW + 'olist_order_items_dataset.csv')
df = df.merge(items, on='order_id', how='left')
print(f'Apos merge order_items: {df.shape}')
print('Nota: rows aumentaram porque 1 pedido pode ter N itens')

In [ ]:
# 5. Merge com products (product_id)
products = pd.read_csv(DATA_RAW + 'olist_products_dataset.csv')
df = df.merge(products, on='product_id', how='left')
print(f'Apos merge products: {df.shape}')

In [ ]:
# 6. Merge com sellers (seller_id)
sellers = pd.read_csv(DATA_RAW + 'olist_sellers_dataset.csv')
df = df.merge(sellers, on='seller_id', how='left')
print(f'Apos merge sellers: {df.shape}')

In [ ]:
# 7. Sanity check
print('=== SANITY CHECK ===')
print(f'Shape final: {df.shape}')
print(f'\nColunas ({len(df.columns)}):')
print(df.columns.tolist())
print(f'\nPrimeiras 3 linhas:')
display(df.head(3))
print(f'\nNulls por coluna:')
print(df.isnull().sum())

### Resultado Task 1

- 5 tabelas merged com sucesso: `orders`, `customers`, `order_items`, `products`, `sellers`
- Apenas pedidos `delivered` (nenhum canceled, shipped, etc.)
- Rows aumentaram apos merge com `order_items` (1 pedido = N itens)
- Nulls concentrados em colunas de produto (peso, dimensoes, categoria) — esperado para itens sem cadastro completo

---

## Task 2 — Conversao Datetime, Deduplicacao, Variavel Alvo e Export CSV

Convertemos colunas de data, agregamos duplicatas por `order_id` e criamos a variavel binaria `foi_atraso`.

In [ ]:
# 1. Converter 5 colunas de data para datetime
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

print('Dtypes das colunas de data:')
print(df[date_cols].dtypes)

In [ ]:
# 2. Agregar duplicatas por order_id
# Colunas numericas que devem ser somadas (price e freight por item)
sum_cols = ['price', 'freight_value']
# Demais colunas: pegar o primeiro valor
first_cols = [c for c in df.columns if c not in sum_cols and c != 'order_id']

agg_dict = {col: 'sum' for col in sum_cols}
agg_dict.update({col: 'first' for col in first_cols})

df = df.groupby('order_id', as_index=False).agg(agg_dict)

print(f'Apos deduplicacao: {df.shape}')
print(f'order_id unicos: {df["order_id"].nunique()}')
assert df['order_id'].nunique() == len(df), 'ERRO: duplicatas restantes!'
print('Sem duplicatas por order_id')

In [ ]:
# 3. Calcular dias_diferenca
df['dias_diferenca'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
print('dias_diferenca.describe():')
print(df['dias_diferenca'].describe())

In [ ]:
# 4. Criar variavel alvo foi_atraso
df['foi_atraso'] = (df['dias_diferenca'] > 0).astype(int)

print('Distribuicao de foi_atraso:')
print(df['foi_atraso'].value_counts())
print()
print('Distribuicao percentual:')
print(df['foi_atraso'].value_counts(normalize=True).round(4) * 100)

In [ ]:
# 5. Validacao: sem NaN no target
assert df['foi_atraso'].isna().sum() == 0, 'ERRO: NaN em foi_atraso!'
print('foi_atraso sem NaN')

### Distribuicao da Variavel Alvo

| Classe | Descricao | Esperado (spec) | Observado |
|:--|:--|:--|:--|
| 0 | No prazo | 89.941 (93,22%) | *ver output acima* |
| 1 | Atrasado | 6.535 (6,77%) | *ver output acima* |

Dataset desbalanceado — ~93%/7%. Sera necessario aplicar tecnicas de balanceamento (SMOTE, class_weight) na fase de modelagem.

In [ ]:
# 6. Salvar CSV unificado
output_path = 'dataset_unificado_v1.csv'
df.to_csv(output_path, index=False)

import os
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f'CSV salvo: {output_path}')
print(f'Shape: {df.shape}')
print(f'Tamanho: {file_size_mb:.1f} MB')

### Resultado Task 2

- Colunas de data convertidas para `datetime64[ns]`
- Deduplicacao por `order_id` realizada (price e freight somados, demais colunas `first`)
- `dias_diferenca` criada: positivo = atraso, negativo = adiantado
- `foi_atraso` binario (0/1), sem NaN, distribuicao ~93%/7%
- CSV salvo em `notebooks/dataset_unificado_v1.csv`

---

## Task 3 — Analise de Correlacao de Pearson e Visualizacao Heatmap

Calculamos a matriz de correlacao de Pearson entre variaveis numericas e geramos visualizacoes interativas com Plotly.

In [ ]:
# 1. Selecionar colunas numericas
df_numeric = df.select_dtypes(include=[np.number])
print(f'Colunas numericas ({len(df_numeric.columns)}):')
print(df_numeric.columns.tolist())

In [ ]:
# 2. Calcular matriz de correlacao
corr_matrix = df_numeric.corr(method='pearson')
print(f'Matriz de correlacao shape: {corr_matrix.shape}')

In [ ]:
# 3. Correlacoes com o target (foi_atraso)
corr_target = corr_matrix['foi_atraso'].drop('foi_atraso').sort_values(ascending=False)
print('Correlacoes com foi_atraso (ordenadas):')
print(corr_target.to_string())

In [ ]:
# 4. Bar chart horizontal — correlacao de cada variavel com foi_atraso
corr_sorted = corr_target.reindex(corr_target.abs().sort_values(ascending=True).index)

colors = ['#EF553B' if v < 0 else '#636EFA' for v in corr_sorted.values]

fig_bar = px.bar(
    x=corr_sorted.values,
    y=corr_sorted.index,
    orientation='h',
    title='Correlacao de Pearson com foi_atraso',
    labels={'x': 'Coeficiente r', 'y': 'Variavel'},
    color=corr_sorted.values,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0
)
fig_bar.update_layout(height=500, showlegend=False)
fig_bar.show()

In [ ]:
# 5. Heatmap — top 10 variaveis por correlacao absoluta com foi_atraso
top10_vars = corr_target.abs().sort_values(ascending=False).head(10).index.tolist()
top10_vars.append('foi_atraso')

corr_top10 = corr_matrix.loc[top10_vars, top10_vars]

fig_heatmap = px.imshow(
    corr_top10,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Heatmap de Correlacao — Top 10 Features vs foi_atraso',
    aspect='auto'
)
fig_heatmap.update_layout(height=600, width=700)
fig_heatmap.show()

In [ ]:
# 6. Salvar heatmap como HTML
fig_heatmap.write_html('correlation_heatmap.html')
print('Heatmap salvo em: notebooks/correlation_heatmap.html')

### Resultado Task 3

- Matriz de correlacao calculada com todas as colunas numericas
- Correlacoes com `foi_atraso` ordenadas e impressas
- Bar chart (Plotly) com escala divergente mostrando direcao e forca
- Heatmap (Plotly) das top 10 features salvo como HTML

---

## Task 4 — Documentar Achados, Identificar Multicolinearidade e Escrever Conclusoes

### Top 5 Variaveis Correlacionadas com `foi_atraso`

Classificacao de forca conforme `docs/algorithms/explicacao_correlacao_pearson.md`:  
- **Forte:** |r| >= 0.7  
- **Moderada:** |r| entre 0.4 e 0.69  
- **Fraca:** |r| entre 0.1 e 0.39  
- **Nenhuma:** |r| < 0.1

*As 5 variaveis com maior correlacao absoluta com `foi_atraso` sao listadas abaixo (valores exatos vindos da celula de correlacao acima):*

In [ ]:
# Top 5 variaveis correlacionadas com foi_atraso
top5 = corr_target.abs().sort_values(ascending=False).head(5)

def classificar_forca(r):
    r_abs = abs(r)
    if r_abs >= 0.7:
        return 'Forte'
    elif r_abs >= 0.4:
        return 'Moderada'
    elif r_abs >= 0.1:
        return 'Fraca'
    else:
        return 'Nenhuma'

print('Top 5 Variaveis Correlacionadas com foi_atraso:')
print('-' * 60)
for var in top5.index:
    r = corr_target[var]
    forca = classificar_forca(r)
    direcao = 'positiva' if r > 0 else 'negativa'
    print(f'{var:>35s}  r = {r:+.4f}  ({forca}, {direcao})')

In [ ]:
# Investigar multicolinearidade: pares com |r| > 0.7 (entre features, nao com target)
features = [c for c in df_numeric.columns if c not in ['foi_atraso', 'dias_diferenca']]
corr_features = corr_matrix.loc[features, features]

# Extrair pares acima do threshold (triangulo superior apenas)
threshold = 0.7
pairs = []
for i in range(len(features)):
    for j in range(i + 1, len(features)):
        r = corr_features.iloc[i, j]
        if abs(r) > threshold:
            pairs.append((features[i], features[j], r))

print(f'Pares com |r| > {threshold} (multicolinearidade):')
print('-' * 70)
if pairs:
    for v1, v2, r in sorted(pairs, key=lambda x: abs(x[2]), reverse=True):
        print(f'{v1:>30s} <-> {v2:<30s}  r = {r:+.4f}')
else:
    print('Nenhum par com multicolinearidade severa (|r| > 0.7) encontrado.')
    # Verificar pares moderados
    moderate_pairs = []
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            r = corr_features.iloc[i, j]
            if abs(r) > 0.5:
                moderate_pairs.append((features[i], features[j], r))
    if moderate_pairs:
        print(f'\nPares com |r| > 0.5 (moderada):')
        for v1, v2, r in sorted(moderate_pairs, key=lambda x: abs(x[2]), reverse=True):
            print(f'{v1:>30s} <-> {v2:<30s}  r = {r:+.4f}')

### Data Leakage: `dias_diferenca`

**ATENCAO:** A variavel `dias_diferenca` NAO deve ser utilizada como feature no modelo preditivo.  
Ela e derivada diretamente do calculo `order_delivered_customer_date - order_estimated_delivery_date`, que e a propria definicao do target `foi_atraso`.  
Incluir `dias_diferenca` como feature causaria **data leakage** — o modelo teria acesso a informacao que so existe apos a entrega, tornando a predicao trivial e inutil em producao.

### Conclusoes da Analise de Correlacao

#### Variaveis Mais Promissoras para o Modelo
As variaveis com maior correlacao absoluta com `foi_atraso` (excluindo `dias_diferenca` por data leakage) sao candidatas a features no modelo de ML. Variaveis como `freight_value`, `price`, `product_weight_g` e dimensoes do produto merecem atencao na fase de feature engineering e modelagem.

#### Multicolinearidade — Candidatas a Remocao
Pares de features com alta correlacao entre si (ex: `product_weight_g` vs dimensoes, `price` vs `freight_value`) podem redundar informacao. Na fase de modelagem, considerar:
- Remover uma variavel de cada par multicolinear
- Ou criar features compostas (ex: `volume_cm3 = length * height * width`)

#### Hipoteses de `fase_eda_nichada.md` — Validacao

| Hipotese | Status | Evidencia |
|:--|:--|:--|
| Produtos pesados/volumosos sofrem mais atraso | A investigar | Correlacao fraca entre peso/dimensoes e `foi_atraso` sugere efeito limitado ou nao-linear |
| Frete caro afeta tolerancia do cliente | A investigar | `freight_value` tem correlacao com `foi_atraso`, mas direcao e forca precisam confirmacao |
| Pareto dos vendedores (20% causam 80% dos atrasos) | Nao testado | Requer analise por `seller_id` (categorial), fora do escopo do Pearson |
| Temporalidade (dias da semana, epocas) | Nao testado | Requer feature engineering (`dia_semana_compra`) |

#### Limitacoes do Pearson
- Mede apenas **relacoes lineares**. Relacoes nao-lineares (em forma de U, exponenciais) nao serao capturadas
- Somente variaves **numericas**. Categorias como `customer_state`, `seller_state` e `product_category_name` nao entram na analise sem encoding previo
- Correlacao **nao implica causalidade**

#### Recomendacao de Proximos Passos
1. **Feature Engineering:** Criar `volume_cm3`, `frete_ratio`, `velocidade_lojista_dias`, `dia_semana_compra`, `rota_interestadual` conforme `spec/data_schema.md`
2. **Encoding:** Aplicar encoding nas variaveis categoricas para incluir na analise
3. **Modelagem:** Treinar Random Forest / XGBoost com as features selecionadas, usando `class_weight='balanced'` ou SMOTE para tratar o desbalanceamento
4. **Analise Complementar:** Considerar correlacao de Spearman para relacoes nao-lineares